# rdfine — a practical tour

`rdfine` provides ergonomic abstractions on top of [rdflib](https://rdflib.readthedocs.io/) and [pyld](https://github.com/digitalbazaar/pyld) for working with RDF. This notebook walks through the three main classes — `PrefixStore`, `GraphReader`, `GraphDict` — plus a couple of utility helpers, using the same catalog the pipeline generator uses.

Run cells top to bottom. Each section is self-contained.

## Setup

Load the demo catalog into an `rdflib.Graph`. Everything downstream operates on this graph.

In [ ]:
from rdflib import Graph
from rdfine import GraphReader, GraphDict, PrefixStore, drop_empty, parse_config

catalog_graph = Graph()
for f in [\"catalog-core.ttl\", \"catalog-ldio.ttl\", \"catalog-rdfc.ttl\", \"catalog-sw.ttl\"]:
    catalog_graph.parse(f"../../data/{f}", publicID="file:///workspace/pipeline/")
print(f"Loaded {len(catalog_graph)} triples")

Loaded 252 triples


## `PrefixStore` — compact, expand, drop

`PrefixStore` is a `prefix -> URL` registry. Build one from a graph and use it to switch string-level between compact CURIE form and full IRIs.

The polymorphic `apply_prefixes(data, action)` works on strings, dicts, lists, and DataFrames — the return type follows the input.

In [2]:
store = PrefixStore(catalog_graph)

# String-level fast paths
url = "https://w3id.org/toolchain#PipelineDefinition"
print("compact :", store.compact_string(url))
print("expand  :", store.expand_string("tcs:PipelineDefinition"))
print("drop    :", store.drop_string("tcs:PipelineDefinition"))

compact : tcs:PipelineDefinition
expand  : https://w3id.org/toolchain#PipelineDefinition
drop    : PipelineDefinition


In [3]:
# Polymorphic apply: same call, different container types
import pandas as pd

data = {
    "type": "https://w3id.org/toolchain#PipelineDefinition",
    "parts": ["https://w3id.org/toolchain#PipelineComponent", "http://purl.org/dc/terms/requires"],
}
print("compacted dict:", store.compact(data))

df = pd.DataFrame({"iri": list(data["parts"])})
print("compacted DataFrame:")
print(store.compact(df))

compacted dict: {'type': 'tcs:PipelineDefinition', 'parts': ['tcs:PipelineComponent', 'dct:requires']}
compacted DataFrame:
                     iri
0  tcs:PipelineComponent
1           dct:requires


## `GraphReader` — a functional view over a graph

`GraphReader` wraps an `rdflib.Graph` and exposes it as an immutable, chainable API. Every transformation returns a new `GraphReader`, so the original is never touched.

The `.df` property gives you a pandas view of all triples with columns `sub`, `pred`, `obj`, `sub_type`, `obj_type`.

In [4]:
reader = GraphReader(catalog_graph)
reader.df.head()

,sub,pred,obj,sub_type,obj_type
0,_:n3a7415f2b1e648e6ab72e61dea88069cb14,tcs:embedded,_:n3a7415f2b1e648e6ab72e61dea88069cb15,<class 'rdflib.term.BNode'>,<class 'rdflib.term.BNode'>
1,sw:deliver-email-service,rdf:type,tcs:PipelineComponent,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>
2,:DeliverEmail,prov:specializationOf,sw:deliver-email-service,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>
3,rdfc:HttpOut,rdfs:label,RDF Connect Http Out,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
4,_:n3a7415f2b1e648e6ab72e61dea88069cb25,rdf:type,tcs:DefaultConfig,<class 'rdflib.term.BNode'>,<class 'rdflib.term.URIRef'>


### `filter` — triple-pattern matching

The most common operation. Any subset of `sub`, `pred`, `obj` narrows the graph; passing a list matches any of the values. The default fast path goes through `rdflib.Graph.triples` for speed; passing `regex=True` (or `sub_type` / `obj_type`) switches to a DataFrame path.

In [5]:
# All pipeline definitions in the catalog
reader.filter(pred="rdf:type", obj="tcs:PipelineDefinition").df

,sub,pred,obj,sub_type,obj_type
0,:DemonstratorPipeline,rdf:type,tcs:PipelineDefinition,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>


In [6]:
# Regex: every subject whose IRI starts with `ldio:`
reader.filter(sub="^ldio:", regex=True).df["sub"].unique()[:10]

array(['ldio:HttpInPoller', 'ldio:RdfAdapter',
       'ldio:LinkedDataInteractionsOrchestrator',
       'ldio:SparqlConstructTransformer', 'ldio:HttpOut',
       'ldio:LdioPipelineStarterService'], dtype=object)

### `select`, `construct`, `ask` — SPARQL

`select` returns a DataFrame, `construct` a new `GraphReader`, `ask` a bool. Prefixes registered on the reader are auto-prepended, so queries can use CURIEs directly.

In [7]:
# List every step of the demonstrator pipeline with the component it specialises
reader.select(
    "?step ?component",
    """
    ?step p-plan:isStepOfPlan :DemonstratorPipeline .
    ?step prov:specializationOf ?component .
    """,
)

,step,component
0,:ApiPoll,ldio:HttpInPoller
1,:RdfConversion,ldio:RdfAdapter
2,:SparqlConstruct,ldio:SparqlConstructTransformer
3,:HttpOut,ldio:HttpOut
4,:HttpIn,rdfc:HttpIn
5,:ThresholdMonitoring,rdfc:thresholdMonitoringProcessor
6,:SparqlIngest,rdfc:SparqlIngest
7,:TriggerAlert,sw:loket-error-alert-service
8,:DeliverEmail,sw:deliver-email-service


In [8]:
# CONSTRUCT: derive a small graph linking each pipeline to its step count
derived = reader.construct(
    "?pipeline :labelledAs ?label .",
    "?pipeline a tcs:PipelineDefinition ; rdfs:label ?label .",
)
derived.df

,sub,pred,obj,sub_type,obj_type
0,:DemonstratorPipeline,:labelledAs,Demonstrator Pipeline.,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>


In [9]:
# ASK: quick boolean check
reader.ask(":DemonstratorPipeline a tcs:PipelineDefinition .")

True

### `traverse` — extract a sub-graph

Recursively follow triples outward from a node. Useful for pulling a self-contained slice of a large graph.

In [10]:
pipeline_slice = reader.traverse(":DemonstratorPipeline")
print(f"Original graph : {len(reader.df):>5} triples")
print(f"Pipeline slice : {len(pipeline_slice.df):>5} triples")

Original graph :   252 triples
Pipeline slice :     3 triples


### `add`, `remove`, `rename` — non-destructive edits

All three return a new `GraphReader`. The original stays intact.

In [11]:
# Rename a node across all subject/object positions
renamed = reader.rename(":DemonstratorPipeline", ":Demo")
renamed.filter(sub=":Demo", pred="rdf:type").df

,sub,pred,obj,sub_type,obj_type
0,:Demo,rdf:type,tcs:PipelineDefinition,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>


### `infer` — forward-chaining rules from a YAML file

Applies a set of `construct` / `where` rules to a fixed point. The pipeline generator uses this to enrich the catalog before compilation.

In [12]:
enriched = reader.infer("../../data/inference_rules.yaml")
print(f"Before inference: {len(reader.df):>5} triples")
print(f"After inference : {len(enriched.df):>5} triples")

Before inference:   252 triples
After inference :   259 triples


## `GraphDict` — a JSON-LD-shaped view

Sometimes you want to work with a graph as a nested dict — for path-based access, framing, or serialization to JSON/YAML. `GraphDict` is that view. It carries its own `PrefixStore` and can round-trip to and from an `rdflib.Graph`.

In [13]:
# Extract a single pipeline step (with its embedded config) and view it as a dict
step_slice = reader.traverse(":ApiPoll").graph
gd = GraphDict(step_slice)
print(gd.serialize("yaml", prefix_action="compact"))

'@graph':
- '@id': ldio:LdioPipelineStarterService
  '@type':
  - tcs:PipelineComponent
  rdfs:label:
  - Ldio Pipeline Starter Service
  tcs:config:
  - '@id': :LdioPipelineStarterDockerComposeConfig
- '@id': :DemonstratorPipeline
  '@type':
  - tcs:PipelineDefinition
  rdfs:comment:
  - Polls API, transforms to RDF, detects threshold, triggers email alert.
  rdfs:label:
  - Demonstrator Pipeline.
- '@id': :LdioPipelineStarterDockerComposeConfig
  '@type':
  - tcs:Config
  - tcs:DockerComposeConfig
  tcs:literal:
  - "\nldio-pipeline-starter:\n    image: curlimages/curl\n    volumes:\n      - ./ldio_pipeline.yml:/pipeline.yml:ro\n\
    \    command: >\n      sh -c \"\n      sleep 30 &&\n      curl -X POST\n     \
    \ -H 'content-type: application/yaml'\n      http://ldio-workbench:8080/admin/api/v1/pipeline\n\
    \      --data-binary @/pipeline.yml\n      \"         \n"
- '@id': ldio:HttpInPoller
  '@type':
  - tcs:PipelineComponent
  ldio:type:
  - Input
  dct:requires:
  - '@id':

### `frame` — reshape via JSON-LD framing

Pin a root node (or type) and the dict is reshaped so that node is at the top.

In [14]:
framed = gd.frame({"@id": ":ApiPoll"})
framed.dict

{'@id': ':ApiPoll',
 '@type': 'tcs:InstancePipelineComponent',
 'p-plan:hasInputVar': {'@type': 'tcs:PipelineConfig',
  'tcs:embedded': {':cron': '*/10 * * * * *',
   ':url': 'https://dishacled-api.azurewebsites.net/api/v1/source-a/current'}},
 'p-plan:isStepOfPlan': {'@id': ':DemonstratorPipeline',
  '@type': 'tcs:PipelineDefinition',
  'rdfs:comment': 'Polls API, transforms to RDF, detects threshold, triggers email alert.',
  'rdfs:label': 'Demonstrator Pipeline.'},
 'prov:specializationOf': {'@id': 'ldio:HttpInPoller',
  '@type': 'tcs:PipelineComponent',
  'ldio:type': 'Input',
  'dct:requires': {'@id': 'ldio:LinkedDataInteractionsOrchestrator',
   '@type': 'tcs:PipelineComponent',
   'dct:requires': {'@id': 'ldio:LdioPipelineStarterService',
    '@type': 'tcs:PipelineComponent',
    'rdfs:label': 'Ldio Pipeline Starter Service',
    'tcs:config': {'@id': ':LdioPipelineStarterDockerComposeConfig',
     '@type': ['tcs:Config', 'tcs:DockerComposeConfig'],
     'tcs:literal': '\r\nldio

### `get`, `set`, `find` — path-based access

Paths can be a `glom.Path`, a list/tuple, or a dot-separated string. `set` is non-creating and returns a new `GraphDict`. `find` regex-searches over a flattened `path` / `value` view.

In [15]:
# Find every path in the framed config whose value looks like a URL
framed.find(value_pattern="^https?://")

,path,value
0,p-plan:hasInputVar.tcs:embedded.:url,https://dishacled-api.azurewebsites.net/api/v1...
1,prov:specializationOf.dct:requires.dcat:landin...,https://informatievlaanderen.github.io/VSDS-Li...
2,prov:specializationOf.dcat:landingPage,https://informatievlaanderen.github.io/VSDS-Li...


## Utilities

`drop_empty` strips falsy values (`None`, `[]`, `{}`, `""`) recursively. Handy after `parse_config` when a schema has many optional keys.

In [16]:
messy = {
    "name": "pipeline",
    "description": "",
    "input": {"adapter": {}, "url": "http://example.org/data"},
    "transformers": [],
    "outputs": [{"name": "console", "config": None}],
}
drop_empty(messy)

{'name': 'pipeline',
 'input': {'url': 'http://example.org/data'},
 'outputs': [{'name': 'console'}]}

## Where to go next

- The full API surface (with all optional parameters) lives in [README.md](README.md).
- For a real-world usage example, see how the pipeline generator's compilers chain `GraphReader` operations in [../compilers/](../compilers/).
- The pipeline generator's own end-to-end demo lives in [../demo.ipynb](../demo.ipynb).